# JS13 Praktikum 2 - ANN Multikelas Sleep Wise
Memprediksi kelas `None`, `Insomnia`, atau `Sleep Apnea`.

In [1]:
from pathlib import Path
import pickle
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

data = pd.read_csv(Path('../Dataset/Sleep_health_and_lifestyle_dataset.csv'))
data['Sleep Disorder'] = data['Sleep Disorder'].fillna('None')
bp = data['Blood Pressure'].str.split('/', expand=True).astype(float)
data['Systolic BP'], data['Diastolic BP'] = bp[0], bp[1]
data = data.drop(columns=['Person ID', 'Blood Pressure'])
X, y = data.drop(columns='Sleep Disorder'), data['Sleep Disorder']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
numeric = X.select_dtypes(include=['number']).columns.tolist()
categorical = [col for col in X.columns if col not in numeric]
preprocess = ColumnTransformer([('numeric', StandardScaler(), numeric), ('categorical', OneHotEncoder(handle_unknown='ignore'), categorical)])
model = Pipeline([('preprocess', preprocess), ('ann', MLPClassifier(hidden_layer_sizes=(64, 32), activation='relu', max_iter=1500, early_stopping=True, random_state=42))])
model.fit(X_train, y_train)
pred = model.predict(X_test)
print(f'Akurasi ANN: {accuracy_score(y_test, pred):.2%}')
print(classification_report(y_test, pred, zero_division=0))
print('Confusion matrix:\n', confusion_matrix(y_test, pred, labels=['None', 'Insomnia', 'Sleep Apnea']))
with open('js13pbl_ann_multiclass.pkl', 'wb') as file: pickle.dump(model, file)

Akurasi ANN: 93.33%
              precision    recall  f1-score   support

    Insomnia       0.86      0.80      0.83        15
        None       1.00      0.95      0.98        44
 Sleep Apnea       0.84      1.00      0.91        16

    accuracy                           0.93        75
   macro avg       0.90      0.92      0.91        75
weighted avg       0.94      0.93      0.93        75

Confusion matrix:
 [[42  2  0]
 [ 0 12  3]
 [ 0  0 16]]
